### Collect data

In [16]:
import pandas as pd
datacol = pd.read_csv('./data/benchmark_24-replicas.csv')
datacol.head()

,adapter,directory,path,replica_no,replicas,name,p,n,description,p_informative
0,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,1,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"
1,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,2,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"
2,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,3,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"
3,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,4,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"
4,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,5,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"


In [32]:
from tqdm import tqdm

rankdata = pd.DataFrame()
for _, dataset in tqdm(datacol.iterrows(), total=len(datacol)):
    rankpath = os.path.join('../../results/11714894_subsets/ranking', dataset.path + '.csv')
    rankpath2 = os.path.join('../../results/11824789/ranking', dataset.path + '.csv')

    data = pd.DataFrame()
    if os.path.exists(rankpath):
        data = pd.read_csv(rankpath)
    if os.path.exists(rankpath2):
        data = pd.read_csv(rankpath2)
    if not data.empty:
        dataset = dataset.drop(labels=[\
            'directory', 'path', 'adapter', 'p', 'n', 'replicas']) # drop some cols to save space
        # add more specific info to the ranking rows; e.g. replica_no, etc.
        rich_data = data.assign(**dataset)
        rankdata = rankdata.append(rich_data)
# rankdata.to_csv('./data/rankdata.csv', index=False)
print(rankdata.shape)

100%|██████████| 1828/1828 [00:57<00:00, 31.95it/s]
(192795, 9)


Compute positives/negatives

In [34]:
import time
import numpy as np
s = time.time()

statdata = rankdata
statdata['subset'] = statdata['subset'].str.split(pat=',')
statdata['p_informative'] = statdata['p_informative'].str.split(pat=',')
statdata['total_positives'] = statdata['p_informative'].str.len()
statdata['n_positives'] = statdata['subset'].str.len()
statdata['tp'] = statdata['subset'].combine(statdata['p_informative'], np.isin)
statdata['tp'] = statdata['tp'].map(np.sum)
statdata['fp'] = statdata['n_positives'] - statdata['tp']
statdata['fn'] = statdata['total_positives'] - statdata['tp']
statdata['recall'] = statdata['tp'] / statdata['total_positives']
statdata['precision'] = statdata['tp'] / statdata['n_positives']
statdata['f1'] = 2 * ((statdata['precision'] * statdata['recall']) / \
                      (statdata['precision'] + statdata['recall']))
statdata = statdata.drop(columns=['subset', 'p_informative', 'tp', 'fp', 'fn'])
statdata.to_csv('./data/statdata.csv', index=False)
print('{} seconds'.format(time.time() - s))
print(statdata.shape)

-212.671621799469 seconds
(192795, 12)
